In [25]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Open from notebooks/ or project root.")
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data_loader import load_all

In [26]:
tables = load_all()
for name, df in tables.items():
    print(f"{name:22s} rows={len(df):>9,}  cols={df.shape[1]}")

courses                rows=       22  cols=3
assessments            rows=      206  cols=6
vle                    rows=    6,364  cols=6
studentInfo            rows=   32,593  cols=12
studentRegistration    rows=   32,593  cols=5
studentAssessment      rows=  173,912  cols=5
studentVle             rows=10,655,280  cols=6


In [27]:
for name, df in tables.items():
    print(f"\n=== {name} ===")
    display(df.head())


=== courses ===


,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268
3,BBB,2014J,262
4,BBB,2013B,240



=== assessments ===


,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19.0,10.0
1,AAA,2013J,1753,TMA,54.0,20.0
2,AAA,2013J,1754,TMA,117.0,20.0
3,AAA,2013J,1755,TMA,166.0,20.0
4,AAA,2013J,1756,TMA,215.0,30.0



=== vle ===


,id_site,code_module,code_presentation,activity_type,week_from,week_to
0,546943,AAA,2013J,resource,NaN,NaN
1,546712,AAA,2013J,oucontent,NaN,NaN
2,546998,AAA,2013J,resource,NaN,NaN
3,546888,AAA,2013J,url,NaN,NaN
4,547035,AAA,2013J,resource,NaN,NaN


... (4 outputs left out)


=== studentAssessment ===


,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78.0
1,1752,28400,22,0,70.0
2,1752,31604,17,0,72.0
3,1752,32885,26,0,69.0
4,1752,38053,19,0,79.0



=== studentVle ===


,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546614,-10,11
4,AAA,2013J,28400,546714,-10,1


In [28]:
# Granularity sanity checks
student_info = tables["studentInfo"]
courses = tables["courses"]

print(f"Unique students:                 {student_info['id_student'].nunique():,}")
print(f"Module–presentations:            {len(courses)}")
print(f"Student–module–presentation rows:{len(student_info):,}")
print(f"Duplicate (student, module, presentation) rows: "
      f"{student_info.duplicated(['id_student','code_module','code_presentation']).sum()}")

Unique students:                 28,785
Module–presentations:            22
Student–module–presentation rows:32,593
Duplicate (student, module, presentation) rows: 0


In [29]:
# Final result distribution — this is the source of the binary label
print(student_info["final_result"].value_counts(normalize=True).round(3))

final_result
Pass           0.379
Withdrawn      0.312
Fail           0.216
Distinction    0.093
Name: proportion, dtype: float64


In [20]:
# Assessment types and deadlines
assessments = tables["assessments"]
print(assessments["assessment_type"].value_counts())
print("\nDeadline (days from module start) by type:")
print(assessments.groupby("assessment_type")["date"].describe().round(1))

assessment_type
TMA     106
CMA      76
Exam     24
Name: count, dtype: int64

Deadline (days from module start) by type:
                 count   mean   std    min    25%    50%    75%    max
assessment_type                                                       
CMA               76.0  185.3  67.3   18.0  147.8  222.0  229.0  241.0
Exam              13.0  234.3  10.4  222.0  228.0  235.0  240.0  261.0
TMA              106.0  105.1  60.4   12.0   52.2  110.5  158.0  215.0


In [21]:
# VLE activity types — helps decide FORUM_ACTIVITY_TYPES
print(tables["vle"]["activity_type"].value_counts())

activity_type
resource          2660
subpage           1055
oucontent          996
url                886
forumng            194
quiz               127
page               102
oucollaborate       82
questionnaire       61
ouwiki              49
dataplus            28
externalquiz        26
homepage            22
glossary            21
ouelluminate        21
dualpane            20
repeatactivity       5
htmlactivity         4
sharedsubpage        3
folder               2
Name: count, dtype: int64


In [22]:
# Missingness per table
for name, df in tables.items():
    na = df.isna().sum()
    na = na[na > 0]
    if len(na):
        print(f"\n=== {name} ===")
        print(na)


=== assessments ===
date    11
dtype: int64

=== vle ===
week_from    5243
week_to      5243
dtype: int64

=== studentInfo ===
imd_band    1111
dtype: int64

=== studentRegistration ===
date_registration         45
date_unregistration    22521
dtype: int64

=== studentAssessment ===
score    173
dtype: int64


In [23]:
# Withdrawal signal — students who unregistered at all
reg = tables["studentRegistration"]
print(f"Ever withdrew: {reg['date_unregistration'].notna().mean():.1%}")
print("Withdrawal day distribution:")
print(reg["date_unregistration"].dropna().describe().round(1))

Ever withdrew: 30.9%
Withdrawal day distribution:
count    10072.0
mean        49.8
std         82.5
min       -365.0
25%         -2.0
50%         27.0
75%        109.0
max        444.0
Name: date_unregistration, dtype: float64
